In [ ]:
!pip install -q qiskit qiskit-aer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 110.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 106.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 96.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 6.9 MB/s eta 0:00:00


In [ ]:
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
import tensorflow as tf
from collections import Counter

from skimage.metrics import (
    mean_squared_error as mse,
    peak_signal_noise_ratio as psnr,
    structural_similarity as ssim
)

from qiskit import QuantumCircuit

In [ ]:
# Load MNIST
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.mnist.load_data()

# Same test image used in conventional FRQI
original = X_test[0]
label = y_test[0]

# Resize to 16×16
image = cv2.resize(
    original,
    (16, 16),
    interpolation=cv2.INTER_AREA
)

print("Image shape:", image.shape)
print("Digit:", label)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Image shape: (16, 16)
Digit: 7


In [ ]:
def adaptive_error_partition(
    image,
    min_size=2,
    max_size=8,
    mse_threshold=500
):

    image_float = image.astype(
        np.float32
    )

    adaptive_patches = []

    def process_region(
        x,
        y,
        size
    ):

        patch = image_float[
            y:y+size,
            x:x+size
        ]

        mean_intensity = np.mean(
            patch
        )

        error = np.mean(
            (
                patch -
                mean_intensity
            ) ** 2
        )

        if (
            error > mse_threshold
            and size > min_size
        ):

            new_size = size // 2

            process_region(
                x,
                y,
                new_size
            )

            process_region(
                x + new_size,
                y,
                new_size
            )

            process_region(
                x,
                y + new_size,
                new_size
            )

            process_region(
                x + new_size,
                y + new_size,
                new_size
            )

        else:

            adaptive_patches.append({

                "x": x,
                "y": y,
                "size": size,
                "mse": error
            })

    for y in range(
        0,
        16,
        max_size
    ):

        for x in range(
            0,
            16,
            max_size
        ):

            process_region(
                x,
                y,
                max_size
            )

    return adaptive_patches

In [ ]:
image_16 = cv2.resize(
    X_test[0],
    (16, 16),
    interpolation=cv2.INTER_AREA
)

adaptive_patches = adaptive_error_partition(
    image_16,
    min_size=2,
    max_size=8,
    mse_threshold=500
)

print("Adaptive Error-Based Patching")
print("-----------------------------")

print(
    "Total patches:",
    len(adaptive_patches)
)

patch_distribution = Counter(
    patch["size"]
    for patch in adaptive_patches
)

for size in [8, 4, 2]:

    print(
        f"{size}×{size} patches:",
        patch_distribution.get(size, 0)
    )

Adaptive Error-Based Patching
-----------------------------
Total patches: 31
8×8 patches: 0
4×4 patches: 11
2×2 patches: 20


In [ ]:
two_value_patches = []

for patch in adaptive_patches:

    x = patch["x"]
    y = patch["y"]
    size = patch["size"]

    patch_image = image_16[
        y:y+size,
        x:x+size
    ]

    if size == 8:

        values = np.array([
            np.mean(
                patch_image[:4, :]
            ),
            np.mean(
                patch_image[4:, :]
            )
        ])

    elif size == 4:

        values = np.array([
            np.mean(
                patch_image[:2, :]
            ),
            np.mean(
                patch_image[2:, :]
            )
        ])

    elif size == 2:

        values = np.array([
            np.mean(
                patch_image[0, :]
            ),
            np.mean(
                patch_image[1, :]
            )
        ])

    two_value_patches.append({

        "x": x,
        "y": y,
        "size": size,
        "values": values,
        "num_values": len(values)
    })

total_values = sum(
    patch["num_values"]
    for patch in two_value_patches
)

print("Two-Value Adaptive Representation")
print("----------------------------------")

print(
    "Total patches:",
    len(two_value_patches)
)

print(
    "Total intensity values:",
    total_values
)

for size in [8, 4, 2]:

    size_patches = [
        p for p in two_value_patches
        if p["size"] == size
    ]

    print(
        f"{size}×{size}:",
        len(size_patches),
        "patches"
    )

    print(
        "Values:",
        sum(
            p["num_values"]
            for p in size_patches
        )
    )

Two-Value Adaptive Representation
----------------------------------
Total patches: 31
Total intensity values: 62
8×8: 0 patches
Values: 0
4×4: 11 patches
Values: 22
2×2: 20 patches
Values: 40


In [ ]:
intensity_values = []

patch_metadata = []

value_index = 0

for patch in two_value_patches:

    x = patch["x"]
    y = patch["y"]
    size = patch["size"]

    values = patch["values"]

    for local_index, value in enumerate(values):

        intensity_values.append(
            float(value)
        )

        patch_metadata.append({

            "index": value_index,

            "x": x,

            "y": y,

            "size": size,

            "local_index": local_index,

            "intensity": float(value)
        })

        value_index += 1

intensity_values = np.array(
    intensity_values,
    dtype=np.float64
)

print("Indexed Two-Value Representation")
print("----------------------------------")

print(
    "Total intensity values:",
    len(intensity_values)
)

print(
    "Minimum intensity:",
    intensity_values.min()
)

print(
    "Maximum intensity:",
    intensity_values.max()
)

print(
    "First 10 intensity values:",
    np.round(
        intensity_values[:10],
        2
    )
)

Indexed Two-Value Representation
----------------------------------
Total intensity values: 62
Minimum intensity: 0.0
Maximum intensity: 214.0
First 10 intensity values: [  0.     0.     0.     0.    15.25   0.   194.5   98.5  104.5  167.5 ]


In [ ]:
theta_values = (
    intensity_values / 255.0
) * (np.pi / 2)

print("FRQI Angle Encoding")
print("-------------------")

print(
    "Total angles:",
    len(theta_values)
)

print(
    "Minimum angle:",
    theta_values.min()
)

print(
    "Maximum angle:",
    theta_values.max()
)

print(
    "First 10 angles:",
    np.round(
        theta_values[:10],
        4
    )
)

FRQI Angle Encoding
-------------------
Total angles: 62
Minimum angle: 0.0
Maximum angle: 1.3182369173886583
First 10 angles: [0.     0.     0.     0.     0.0939 0.     1.1981 0.6068 0.6437 1.0318]


In [ ]:
num_values = len(
    intensity_values
)

position_qubits_count = int(
    np.ceil(
        np.log2(num_values)
    )
)

intensity_qubit = 0

position_qubits = list(
    range(
        1,
        position_qubits_count + 1
    )
)

total_qubits = (
    position_qubits_count + 1
)

print("Multi-Value Adaptive-FRQI Resources")
print("------------------------------------")

print(
    "Intensity values:",
    num_values
)

print(
    "Position qubits:",
    position_qubits_count
)

print(
    "Intensity qubits:",
    1
)

print(
    "Total qubits:",
    total_qubits
)

print(
    "Available position states:",
    2 ** position_qubits_count
)

print(
    "Unused position states:",
    (2 ** position_qubits_count) - num_values
)

Multi-Value Adaptive-FRQI Resources
------------------------------------
Intensity values: 62
Position qubits: 6
Intensity qubits: 1
Total qubits: 7
Available position states: 64
Unused position states: 2


In [ ]:
adaptive_qc = QuantumCircuit(
    total_qubits
)

for q in position_qubits:

    adaptive_qc.h(q)

for index, angle in enumerate(
    theta_values
):

    binary = format(
        index,
        f"0{position_qubits_count}b"
    )

    binary = binary[::-1]

    for q, bit in zip(
        position_qubits,
        binary
    ):

        if bit == "0":

            adaptive_qc.x(q)

    adaptive_qc.mcry(
        2 * angle,
        position_qubits,
        intensity_qubit
    )

    for q, bit in zip(
        position_qubits,
        binary
    ):

        if bit == "0":

            adaptive_qc.x(q)

print("Multi-Value Adaptive-FRQI Circuit")
print("----------------------------------")

print(
    "Total qubits:",
    adaptive_qc.num_qubits
)

print(
    "Total gates:",
    adaptive_qc.size()
)

print(
    "Circuit depth:",
    adaptive_qc.depth()
)

Multi-Value Adaptive-FRQI Circuit
----------------------------------
Total qubits: 7
Total gates: 8324
Circuit depth: 4723


In [ ]:
print(adaptive_qc.draw(
    "text",
    fold=120
))

     ┌───┐┌────────┐                                                                                   ┌───┐┌─────────┐»
q_0: ┤ H ├┤ P(π/8) ├───────────────────────────────────────────────────────────────────────────────────┤ X ├┤ P(-π/8) ├»
     ├───┤└─┬───┬──┘┌────────┐                                                                         └─┬─┘└─────────┘»
q_1: ┤ H ├──┤ X ├───┤ P(π/8) ├──■───────────────■────────────────────■──────────────────────────────■────┼─────────────»
     ├───┤  ├───┤   ├────────┤┌─┴─┐┌─────────┐┌─┴─┐                  │                              │    │             »
q_2: ┤ H ├──┤ X ├───┤ P(π/8) ├┤ X ├┤ P(-π/8) ├┤ X ├──■───────────────┼──────────────■───────────────┼────┼─────────────»
     ├───┤  ├───┤   ├────────┤└───┘└─────────┘└───┘┌─┴─┐┌─────────┐┌─┴─┐┌────────┐┌─┴─┐┌─────────┐┌─┴─┐  │             »
q_3: ┤ H ├──┤ X ├───┤ P(π/8) ├─────────────────────┤ X ├┤ P(-π/8) ├┤ X ├┤ P(π/8) ├┤ X ├┤ P(-π/8) ├┤ X ├──■─────────────»
     ├───┤  ├───┤   ├────────┤  

In [ ]:
from qiskit.quantum_info import Statevector

statevector = Statevector.from_instruction(
    adaptive_qc
)

print("Statevector simulation completed.")

print(
    "Number of amplitudes:",
    len(statevector.data)
)

Statevector simulation completed.
Number of amplitudes: 128


In [ ]:
reconstructed_values = []

num_positions = 2 ** position_qubits_count

for index in range(num_values):

    state_index = (
        (index << 1) + 1
    )

    probability = (
        abs(
            statevector.data[state_index]
        ) ** 2
    )

    theta = np.arcsin(
        np.sqrt(
            probability * num_positions
        )
    )

    intensity = (
        theta / (np.pi / 2)
    ) * 255.0

    reconstructed_values.append(
        intensity
    )

reconstructed_values = np.array(
    reconstructed_values
)

print("Quantum Intensity Extraction")
print("----------------------------")

print(
    "Extracted values:",
    len(reconstructed_values)
)

print(
    "Minimum intensity:",
    reconstructed_values.min()
)

print(
    "Maximum intensity:",
    reconstructed_values.max()
)

print(
    "First 10 extracted values:",
    np.round(
        reconstructed_values[:10],
        2
    )
)

Quantum Intensity Extraction
----------------------------
Extracted values: 62
Minimum intensity: 8.6303190649452e-14
Maximum intensity: 213.99999999997047
First 10 extracted values: [  0.     0.     0.     0.    15.25   0.   194.5   98.5  104.5  167.5 ]


In [ ]:
quantum_reconstructed = np.zeros(
    (16, 16),
    dtype=np.float64
)

value_counter = 0

for patch in two_value_patches:

    x = patch["x"]
    y = patch["y"]
    size = patch["size"]

    if size == 8:

        quantum_reconstructed[
            y:y+4,
            x:x+8
        ] = reconstructed_values[
            value_counter
        ]

        quantum_reconstructed[
            y+4:y+8,
            x:x+8
        ] = reconstructed_values[
            value_counter + 1
        ]

        value_counter += 2

    elif size == 4:

        quantum_reconstructed[
            y:y+2,
            x:x+4
        ] = reconstructed_values[
            value_counter
        ]

        quantum_reconstructed[
            y+2:y+4,
            x:x+4
        ] = reconstructed_values[
            value_counter + 1
        ]

        value_counter += 2

    elif size == 2:

        quantum_reconstructed[
            y:y+1,
            x:x+2
        ] = reconstructed_values[
            value_counter
        ]

        quantum_reconstructed[
            y+1:y+2,
            x:x+2
        ] = reconstructed_values[
            value_counter + 1
        ]

        value_counter += 2

print("Two-Value Quantum Reconstruction")
print("----------------------------------")

print(
    "Values mapped:",
    value_counter
)

Two-Value Quantum Reconstruction
----------------------------------
Values mapped: 62


In [ ]:
quantum_mse = mse(
    image_16,
    quantum_reconstructed
)

quantum_psnr = psnr(
    image_16,
    quantum_reconstructed,
    data_range=255
)

quantum_ssim = ssim(
    image_16,
    quantum_reconstructed,
    data_range=255
)

print("Multi-Value Adaptive-FRQI Reconstruction")
print("-----------------------------------------")

print(
    "MSE  :",
    quantum_mse
)

print(
    "PSNR :",
    quantum_psnr,
    "dB"
)

print(
    "SSIM :",
    quantum_ssim
)

Multi-Value Adaptive-FRQI Reconstruction
-----------------------------------------
MSE  : 472.87646484375
PSNR : 21.383326612378536 dB
SSIM : 0.8893954629741727


In [ ]:
two_value_frqi_results = []

num_images = 100

for idx in range(num_images):

    image_16 = cv2.resize(
        X_test[idx],
        (16, 16),
        interpolation=cv2.INTER_AREA
    )

    adaptive_patches = adaptive_error_partition(
        image_16,
        min_size=2,
        max_size=8,
        mse_threshold=500
    )

    two_value_patches = []

    for patch in adaptive_patches:

        x = patch["x"]
        y = patch["y"]
        size = patch["size"]

        patch_image = image_16[
            y:y+size,
            x:x+size
        ]

        if size == 8:

            values = np.array([
                np.mean(
                    patch_image[:4, :]
                ),
                np.mean(
                    patch_image[4:, :]
                )
            ])

        elif size == 4:

            values = np.array([
                np.mean(
                    patch_image[:2, :]
                ),
                np.mean(
                    patch_image[2:, :]
                )
            ])

        elif size == 2:

            values = np.array([
                np.mean(
                    patch_image[0, :]
                ),
                np.mean(
                    patch_image[1, :]
                )
            ])

        two_value_patches.append({

            "x": x,
            "y": y,
            "size": size,
            "values": values
        })

    intensity_values_temp = []

    for patch in two_value_patches:

        for value in patch["values"]:

            intensity_values_temp.append(
                float(value)
            )

    intensity_values_temp = np.array(
        intensity_values_temp,
        dtype=np.float64
    )

    num_values = len(
        intensity_values_temp
    )

    theta_values_temp = (
        intensity_values_temp / 255.0
    ) * (np.pi / 2)

    position_qubits_count_temp = int(
        np.ceil(
            np.log2(num_values)
        )
    )

    intensity_qubit_temp = 0

    position_qubits_temp = list(
        range(
            1,
            position_qubits_count_temp + 1
        )
    )

    total_qubits_temp = (
        position_qubits_count_temp + 1
    )

    adaptive_qc_temp = QuantumCircuit(
        total_qubits_temp
    )

    for q in position_qubits_temp:

        adaptive_qc_temp.h(q)

    for value_index, angle in enumerate(
        theta_values_temp
    ):

        binary = format(
            value_index,
            f"0{position_qubits_count_temp}b"
        )

        binary = binary[::-1]

        for q, bit in zip(
            position_qubits_temp,
            binary
        ):

            if bit == "0":

                adaptive_qc_temp.x(q)

        adaptive_qc_temp.mcry(
            2 * angle,
            position_qubits_temp,
            intensity_qubit_temp
        )

        for q, bit in zip(
            position_qubits_temp,
            binary
        ):

            if bit == "0":

                adaptive_qc_temp.x(q)

    statevector_temp = Statevector.from_instruction(
        adaptive_qc_temp
    )

    reconstructed_values_temp = []

    num_positions_temp = (
        2 ** position_qubits_count_temp
    )

    for value_index in range(
        num_values
    ):

        state_index = (
            (value_index << 1) + 1
        )

        probability = (
            abs(
                statevector_temp.data[
                    state_index
                ]
            ) ** 2
        )

        theta = np.arcsin(
            np.sqrt(
                probability *
                num_positions_temp
            )
        )

        intensity = (
            theta / (np.pi / 2)
        ) * 255.0

        reconstructed_values_temp.append(
            intensity
        )

    reconstructed_values_temp = np.array(
        reconstructed_values_temp
    )

    quantum_reconstructed_temp = np.zeros(
        (16, 16),
        dtype=np.float64
    )

    value_counter = 0

    for patch in two_value_patches:

        x = patch["x"]
        y = patch["y"]
        size = patch["size"]

        if size == 8:

            quantum_reconstructed_temp[
                y:y+4,
                x:x+8
            ] = reconstructed_values_temp[
                value_counter
            ]

            quantum_reconstructed_temp[
                y+4:y+8,
                x:x+8
            ] = reconstructed_values_temp[
                value_counter + 1
            ]

            value_counter += 2

        elif size == 4:

            quantum_reconstructed_temp[
                y:y+2,
                x:x+4
            ] = reconstructed_values_temp[
                value_counter
            ]

            quantum_reconstructed_temp[
                y+2:y+4,
                x:x+4
            ] = reconstructed_values_temp[
                value_counter + 1
            ]

            value_counter += 2

        elif size == 2:

            quantum_reconstructed_temp[
                y:y+1,
                x:x+2
            ] = reconstructed_values_temp[
                value_counter
            ]

            quantum_reconstructed_temp[
                y+1:y+2,
                x:x+2
            ] = reconstructed_values_temp[
                value_counter + 1
            ]

            value_counter += 2

    mse_value = mse(
        image_16,
        quantum_reconstructed_temp
    )

    psnr_value = psnr(
        image_16,
        quantum_reconstructed_temp,
        data_range=255
    )

    ssim_value = ssim(
        image_16,
        quantum_reconstructed_temp,
        data_range=255
    )

    two_value_frqi_results.append({

        "Image": idx,

        "Adaptive Patches": len(
            two_value_patches
        ),

        "Intensity Values": num_values,

        "Qubits": adaptive_qc_temp.num_qubits,

        "Gates": adaptive_qc_temp.size(),

        "Depth": adaptive_qc_temp.depth(),

        "MSE": mse_value,

        "PSNR": psnr_value,

        "SSIM": ssim_value
    })

    print(
        f"Image {idx+1}/{num_images} completed",
        end="\r"
    )

two_value_frqi_results = pd.DataFrame(
    two_value_frqi_results
)

print(
    "\n\n100-image Two-Value Adaptive-FRQI evaluation completed."
)

display(
    two_value_frqi_results.head()
)

Image 100/100 completed

100-image Two-Value Adaptive-FRQI evaluation completed.


,Image,Adaptive Patches,Intensity Values,Qubits,Gates,Depth,MSE,PSNR,SSIM
0,0,31,62,7,8324,4723,472.876465,21.383327,0.889395
1,1,46,92,8,19311,16009,646.505859,20.025079,0.906426
2,2,28,56,7,7534,4267,504.689819,21.100558,0.749256
3,3,46,92,8,19311,16009,807.425781,19.059777,0.920881
4,4,37,74,8,15557,12877,1077.422363,17.806944,0.791569


In [ ]:
display(
    two_value_frqi_results.head()
)

,Image,Adaptive Patches,Intensity Values,Qubits,Gates,Depth,MSE,PSNR,SSIM
0,0,31,62,7,8324,4723,472.876465,21.383327,0.889395
1,1,46,92,8,19311,16009,646.505859,20.025079,0.906426
2,2,28,56,7,7534,4267,504.689819,21.100558,0.749256
3,3,46,92,8,19311,16009,807.425781,19.059777,0.920881
4,4,37,74,8,15557,12877,1077.422363,17.806944,0.791569


In [ ]:
finite_psnr = two_value_frqi_results[
    np.isfinite(
        two_value_frqi_results["PSNR"]
    )
]["PSNR"]

exact_reconstructions = (
    two_value_frqi_results["MSE"] < 1e-10
).sum()

summary = pd.DataFrame({

    "Metric": [
        "Average Adaptive Patches",
        "Average Intensity Values",
        "Average Qubits",
        "Average Gates",
        "Average Depth",
        "Average MSE",
        "Average Finite PSNR",
        "Average SSIM",
        "Exact Reconstructions"
    ],

    "Value": [
        two_value_frqi_results[
            "Adaptive Patches"
        ].mean(),

        two_value_frqi_results[
            "Intensity Values"
        ].mean(),

        two_value_frqi_results[
            "Qubits"
        ].mean(),

        two_value_frqi_results[
            "Gates"
        ].mean(),

        two_value_frqi_results[
            "Depth"
        ].mean(),

        two_value_frqi_results[
            "MSE"
        ].mean(),

        finite_psnr.mean(),

        two_value_frqi_results[
            "SSIM"
        ].mean(),

        exact_reconstructions
    ]
})

summary["Value"] = summary[
    "Value"
].round(3)

display(summary)

,Metric,Value
0,Average Adaptive Patches,38.320
1,Average Intensity Values,76.640
2,Average Qubits,7.830
3,Average Gates,15366.230
4,Average Depth,12387.460
5,Average MSE,741.616
6,Average Finite PSNR,19.635
7,Average SSIM,0.856
8,Exact Reconstructions,0.000


In [ ]:
print("Quantum Resource Range")
print("---------------------")

print(
    "Qubits:",
    two_value_frqi_results["Qubits"].min(),
    "to",
    two_value_frqi_results["Qubits"].max()
)

print(
    "Gates:",
    two_value_frqi_results["Gates"].min(),
    "to",
    two_value_frqi_results["Gates"].max()
)

print(
    "Depth:",
    two_value_frqi_results["Depth"].min(),
    "to",
    two_value_frqi_results["Depth"].max()
)

print(
    "Intensity Values:",
    two_value_frqi_results[
        "Intensity Values"
    ].min(),
    "to",
    two_value_frqi_results[
        "Intensity Values"
    ].max()
)

Quantum Resource Range
---------------------
Qubits: 7 to 8
Gates: 6740 to 21807
Depth: 3811 to 18097
Intensity Values: 50 to 104


In [ ]:
from google.colab import files

two_value_frqi_results.to_csv(
    "two_value_adaptive_frqi_100_images.csv",
    index=False
)

files.download(
    "two_value_adaptive_frqi_100_images.csv"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>